# Laboratorio 1: Dinámica y Estabilidad Numérica en el Modelo Logístico 1D

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nabw/UNA-taller/blob/main/notebooks/estudiante/01_modelo_logistico.ipynb)


## 1. Introducción y Contexto Matemático

El modelo logístico, propuesto originalmente por Pierre François Verhulst en 1838, describe la dinámica temporal de una población en un entorno con recursos finitos. En su formulación normalizada, la fracción de la población respecto a la capacidad de carga, $x(t) \in [0, 1]$, satisface la ecuación diferencial ordinaria (EDO):

$$\frac{dx}{dt} = C x (1 - x)$$

donde:
- $x(t)$ es la variable de estado adimensional.
- $C > 0$ representa la tasa intrínseca de crecimiento per cápita.
- El término $C x$ modela la expansión exponencial no restringida.
- El factor $(1 - x)$ modela la saturación por competencia ambiental (densodependencia).

### Puntos de Equilibrio y Comportamiento Asintótico

Los puntos críticos corresponden a las raíces de $f(x) = C x (1 - x) = 0$:
- $x^* = 0$: punto de equilibrio inestable ($f'(0) = C > 0$).
- $x^* = 1$: punto de equilibrio asintóticamente estable ($f'(1) = -C < 0$).

Para cualquier condición inicial $x_0 \in (0, 1)$, la solución continua es estrictamente creciente, acotada superiormente por 1 y libre de oscilaciones.

## Objetivos de Aprendizaje

1. Deducir e implementar la solución analítica exacta en forma cerrada.
2. Implementar e integrar la trayectoria mediante el método de Euler explícito.
3. Implementar el método de Euler implícito resolviendo analíticamente la raíz cuadrática admisible.
4. Implementar el esquema Implícito-Explícito (IMEX) linealizado.
5. Comparar la estabilidad numérica en dos regímenes temporales: paso fino ($\Delta t = 0.1$) frente a paso crítico ($\Delta t = 1.3$) con $C = 2.0$.


In [ ]:
# Celda de configuración y librerías científicas
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
import numpy as np
import matplotlib.pyplot as plt

# Estilo sobrio y legible para figuras científicas
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "figure.titlesize": 13,
    "lines.linewidth": 2.0,
    "grid.alpha": 0.4
})

print("Librerías importadas correctamente.")


## 2. Solución Analítica Exacta

La ecuación logística es separable:

$$\frac{dx}{x (1 - x)} = C dt$$

Descomponiendo en fracciones parciales:

$$\int \left( \frac{1}{x} + \frac{1}{1 - x} \right) dx = \int C dt \implies \ln\left( \frac{x}{1 - x} \right) = C t + K$$

Aplicando la condición inicial $x(0) = x_0$ y despejando $x(t)$ de forma numéricamente estable:

$$x(t) = \frac{1}{1 + \left( \frac{1 - x_0}{x_0} \right) e^{-C t}}$$

### Especificación de la Función a Implementar
- **Firma:** `solucion_exacta(x0: float, C: float, t: np.ndarray) -> np.ndarray`
- **Parámetros de entrada:**
  - `x0` (`float`): Condición inicial $x(0) \in (0, 1)$.
  - `C` (`float`): Tasa intrínseca de crecimiento per cápita ($C > 0$).
  - `t` (`np.ndarray`): Vector unidimensional de instantes de tiempo.
- **Salida esperada:**
  - `np.ndarray`: Arreglo unidimensional con los valores exactos de $x(t)$ de igual longitud que `t`.


In [ ]:
def solucion_exacta(x0: float, C: float, t: np.ndarray) -> np.ndarray:
    """
    Calcula la solución analítica exacta de la ecuación logística 1D.

    Parámetros
    ----------
    x0 : float
        Condición inicial x(0), con x0 en (0, 1).
    C : float
        Tasa intrínseca de crecimiento per cápita (C > 0).
    t : np.ndarray
        Vector unidimensional de instantes de tiempo.

    Retorna
    -------
    np.ndarray
        Valores exactos de x(t).
    """
    # TU CÓDIGO AQUÍ
    ...


In [ ]:
# Pruebas de validación para solucion_exacta
t_test = np.array([0.0, 1.0, 5.0, 15.0])
x_test = solucion_exacta(0.1, 2.0, t_test)

assert isinstance(x_test, np.ndarray), "La función debe retornar un arreglo de NumPy."
assert x_test.shape == t_test.shape, "La forma del arreglo de salida debe ser igual a la de t."
assert np.isclose(x_test[0], 0.1), f"x(0) debe ser 0.1, obtenido {x_test[0]:.4f}"
assert np.isclose(x_test[-1], 1.0, atol=1e-5), f"Para t grande, x debe tender a 1.0, obtenido {x_test[-1]:.6f}"
assert np.all(np.diff(x_test) > 0), "La solución logística continua con x0 en (0, 1) debe ser estrictamente creciente."
print("Validación de solucion_exacta: EXITOSA.")


## 3. Método de Euler Explícito

Discretizamos el intervalo $[0, T]$ en pasos regulares $t_n = n \Delta t$. Reemplazando la derivada temporal por una diferencia finita hacia adelante:

$$\frac{x_{n+1} - x_n}{\Delta t} = C x_n (1 - x_n)$$

Despejando el estado en el siguiente paso:

$$x_{n+1} = x_n + \Delta t C x_n (1 - x_n)$$

### Condición de Estabilidad del Método Explícito

A diferencia de la ecuación diferencial continua, el método de Euler explícito impone una cota estricta sobre el paso temporal para evitar oscilaciones espurias o divergencia numérica:

$$\Delta t < \frac{2}{C}$$

- Si $\Delta t < \frac{1}{C}$: la solución numérica converge de forma monótona hacia el equilibrio $x^* = 1$.
- Si $\frac{1}{C} < \Delta t < \frac{2}{C}$: la solución presenta oscilaciones transitorias pero se amortigua hacia 1.
- Si $\Delta t > \frac{2}{C}$: el método es inestable, produciendo sobrepasos no físicos ($x > 1$) y oscilaciones espurias de período 2.

Para el análisis formal de estabilidad lineal mediante la ecuación de prueba de Dahlquist y el factor de amplificación, consulta el [Apéndice Teórico: Estabilidad Temporal](https://github.com/nabw/UNA-taller/blob/main/appendix/estabilidad_temporal.md).

### Especificación de la Función a Implementar
- **Firma:** `euler_explicito(x0: float, C: float, dt: float, T: float) -> tuple[np.ndarray, np.ndarray]`
- **Parámetros de entrada:**
  - `x0` (`float`): Condición inicial $x(0)$.
  - `C` (`float`): Tasa intrínseca de crecimiento per cápita.
  - `dt` (`float`): Tamaño del paso temporal $\Delta t$.
  - `T` (`float`): Tiempo final de integración $T$.
- **Salida esperada:**
  - `tuple[np.ndarray, np.ndarray]`: Tupla `(t, x)` donde `t` es la malla temporal discreta $[0, \Delta t, \dots, N\Delta t]$ y `x` es el vector con la solución calculada en cada instante temporal, ambos de longitud $N + 1$ con $N = \text{round}(T / \Delta t)$.


In [ ]:
def euler_explicito(
    x0: float,
    C: float,
    dt: float,
    T: float
) -> tuple[np.ndarray, np.ndarray]:
    """
    Resuelve la ecuación logística mediante el esquema de Euler explícito.

    Parámetros
    ----------
    x0 : float
        Condición inicial.
    C : float
        Parámetro de crecimiento.
    dt : float
        Tamaño del paso temporal.
    T : float
        Tiempo final.

    Retorna
    -------
    t : np.ndarray
        Malla temporal discreta.
    x : np.ndarray
        Valores discretos de la solución.
    """
    # TU CÓDIGO AQUÍ
    ...


In [ ]:
# Pruebas de validación para euler_explicito
t_exp, x_exp = euler_explicito(0.1, 2.0, 0.1, 1.0)
assert len(t_exp) == 11, f"Longitud esperada: 11, obtenida: {len(t_exp)}"
assert np.isclose(x_exp[0], 0.1), "La condición inicial no coincide."
expected_step1 = 0.1 + 0.1 * 2.0 * 0.1 * (1.0 - 0.1)
assert np.isclose(x_exp[1], expected_step1), f"Paso 1 esperado: {expected_step1}, obtenido: {x_exp[1]}"
print("Validación de euler_explicito: EXITOSA.")


## 4. Método de Euler Implícito (Backward Euler)

Evaluamos la tasa de variación en el instante futuro $t_{n+1}$:

$$\frac{x_{n+1} - x_n}{\Delta t} = C x_{n+1} (1 - x_{n+1})$$

Multiplicando por $\Delta t$ y agrupando términos:

$$(\Delta t C) x_{n+1}^2 + (1 - \Delta t C) x_{n+1} - x_n = 0$$

Esta es una ecuación cuadrática de la forma $a y^2 + b y + c = 0$, con:
- $a = \Delta t C > 0$
- $b = 1 - \Delta t C$
- $c = -x_n$

El discriminante es $\Delta = b^2 - 4ac = (1 - \Delta t C)^2 + 4 \Delta t C x_n \ge 0$ para todo $x_n \ge 0$. Puesto que $c = -x_n \le 0$ y $a > 0$, el producto de las dos raíces es $c/a \le 0$. Por lo tanto, existe una única raíz real no negativa (físicamente admisible):

$$x_{n+1} = \frac{-(1 - \Delta t C) + \sqrt{(1 - \Delta t C)^2 + 4 \Delta t C x_n}}{2 \Delta t C}$$

Para evitar cancelación sustractiva cuando $b > 0$, la raíz se puede formular de manera equivalente como:

$$x_{n+1} = \frac{2 x_n}{b + \sqrt{b^2 + 4 a x_n}}$$

> **Nota pedagógica:** Ambas expresiones son algebraicamente equivalentes; puedes implementar cualquiera de las dos (o la selección condicional según el signo de $b = 1 - \Delta t C$) para superar las pruebas de validación.

### Especificación de la Función a Implementar
- **Firma:** `euler_implicito(x0: float, C: float, dt: float, T: float) -> tuple[np.ndarray, np.ndarray]`
- **Parámetros de entrada:**
  - `x0` (`float`): Condición inicial $x(0)$.
  - `C` (`float`): Tasa intrínseca de crecimiento per cápita.
  - `dt` (`float`): Tamaño del paso temporal $\Delta t$.
  - `T` (`float`): Tiempo final de integración $T$.
- **Salida esperada:**
  - `tuple[np.ndarray, np.ndarray]`: Tupla `(t, x)` donde `t` es la malla temporal discreta y `x` es el vector numérico calculado mediante la raíz cuadrática admisible, ambos de longitud $N + 1$.


In [ ]:
def euler_implicito(
    x0: float,
    C: float,
    dt: float,
    T: float
) -> tuple[np.ndarray, np.ndarray]:
    """
    Resuelve la ecuación logística mediante el esquema de Euler implícito,
    utilizando la raíz analítica positiva de la ecuación cuadrática.
    """
    # TU CÓDIGO AQUÍ
    ...


In [ ]:
# Pruebas de validación para euler_implicito
t_imp, x_imp = euler_implicito(0.1, 2.0, 0.1, 1.0)
assert len(t_imp) == 11, f"Longitud esperada: 11, obtenida: {len(t_imp)}"
assert np.isclose(x_imp[0], 0.1), "La condición inicial no coincide."
# Verificar que satisface la relación algebraica implícita en cada paso
residuos = x_imp[1:] - x_imp[:-1] - 0.1 * 2.0 * x_imp[1:] * (1.0 - x_imp[1:])
assert np.all(np.abs(residuos) < 1e-12), f"Residuo máximo: {np.max(np.abs(residuos)):.2e}"
print("Validación de euler_implicito: EXITOSA.")


## 5. Esquema Implícito-Explícito (IMEX) Linealizado

Para evitar la resolución de polinomios o métodos iterativos no lineales, podemos desacoplar la dinámica tratando el término lineal de expansión de manera explícita y el término no lineal de saturación de manera implícita:

$$\frac{x_{n+1} - x_n}{\Delta t} = C x_n - C x_n x_{n+1}$$

Reorganizando la expresión algebraicamente:

$$x_{n+1} (1 + \Delta t C x_n) = x_n (1 + \Delta t C)$$

Despejando directamente $x_{n+1}$:

$$x_{n+1} = \frac{x_n (1 + \Delta t C)}{1 + \Delta t C x_n}$$

### Propiedades Analíticas Fundamentales
1. **Linealidad implícita**: El cálculo de $x_{n+1}$ es una operación aritmética directa.
2. **Positividad incondicional**: Si $x_n \ge 0$, entonces tanto el numerador como el denominador son estrictamente positivos para cualquier $\Delta t > 0$, garantizando $x_{n+1} \ge 0$.
3. **Conservación del punto fijo**: Si $x_n = 1$, entonces $x_{n+1} = \frac{1 + \Delta t C}{1 + \Delta t C} = 1$ de forma exacta para todo $\Delta t$.

### Especificación de la Función a Implementar
- **Firma:** `imex_linealizado(x0: float, C: float, dt: float, T: float) -> tuple[np.ndarray, np.ndarray]`
- **Parámetros de entrada:**
  - `x0` (`float`): Condición inicial $x(0)$.
  - `C` (`float`): Tasa intrínseca de crecimiento per cápita.
  - `dt` (`float`): Tamaño del paso temporal $\Delta t$.
  - `T` (`float`): Tiempo final de integración $T$.
- **Salida esperada:**
  - `tuple[np.ndarray, np.ndarray]`: Tupla `(t, x)` donde `t` es la malla temporal discreta y `x` es el vector numérico calculado mediante la actualización algebraica directa de IMEX, ambos de longitud $N + 1$.


In [ ]:
def imex_linealizado(
    x0: float,
    C: float,
    dt: float,
    T: float
) -> tuple[np.ndarray, np.ndarray]:
    """
    Resuelve la ecuación logística mediante el esquema IMEX linealizado.
    """
    # TU CÓDIGO AQUÍ
    ...


In [ ]:
# Pruebas de validación para imex_linealizado
t_imx, x_imx = imex_linealizado(0.1, 2.0, 0.1, 1.0)
assert len(t_imx) == 11, f"Longitud esperada: 11, obtenida: {len(t_imx)}"
assert np.isclose(x_imx[0], 0.1), "La condición inicial no coincide."
expected_p1 = (0.1 * (1.0 + 0.1 * 2.0)) / (1.0 + 0.1 * 2.0 * 0.1)
assert np.isclose(x_imx[1], expected_p1), f"Paso 1 esperado: {expected_p1}, obtenido: {x_imx[1]}"
assert np.all(x_imx >= 0.0), "El esquema IMEX debe preservar no-negatividad."
print("Validación de imex_linealizado: EXITOSA.")


## 6. Comparación de Regímenes Temporales: Paso Estable vs Inestable

Fijamos los parámetros físicos:
- Tasa de crecimiento: $C = 2.0$
- Condición inicial: $x_0 = 0.05$
- Horizonte de integración: $T = 8.0$

Evaluamos dos regímenes de paso temporal:
1. **Paso Fino (Régimen Asintótico)**: $\Delta t = 0.1$.
   $$\Delta t C = 0.2 < 2.0$$
   Dentro del radio de estabilidad lineal para todos los esquemas.

2. **Paso Crítico (Régimen Inestable para Explícito)**: $\Delta t = 1.3$.
   $$\Delta t C = 2.6 > 2.0$$
   Euler explícito supera el umbral crítico de estabilidad lineal, produciendo oscilaciones espurias y sobrepaso de la capacidad de carga ($x > 1$).


In [ ]:
# Parámetros de simulación
C = 2.0
x0 = 0.05
T = 8.0

dt_cases = [0.1, 1.3]
results = {}

print("=" * 75)
print(f"{'dt':>6s} | {'Método':>16s} | {'Error L_inf':>12s} | {'Máximo x':>10s} | {'Sobrepaso x>1':>14s}")
print("=" * 75)

for dt in dt_cases:
    results[dt] = {}
    # Referencia exacta
    t_eval = np.linspace(0.0, T, int(np.round(T / dt)) + 1)
    x_exact = solucion_exacta(x0, C, t_eval)
    results[dt]["exact"] = (t_eval, x_exact)

    # Integradores numéricos
    t_exp, x_exp = euler_explicito(x0, C, dt, T)
    t_imp, x_imp = euler_implicito(x0, C, dt, T)
    t_imx, x_imx = imex_linealizado(x0, C, dt, T)

    for name, t_num, x_num in [
        ("Euler Explícito", t_exp, x_exp),
        ("Euler Implícito", t_imp, x_imp),
        ("IMEX Linealizado", t_imx, x_imx)
    ]:
        err_linf = float(np.max(np.abs(x_num - x_exact)))
        max_val = float(np.max(x_num))
        overshoot = "SÍ" if max_val > 1.0 + 1e-5 else "NO"
        results[dt][name] = (t_num, x_num, err_linf, max_val)
        print(f"{dt:6.2f} | {name:>16s} | {err_linf:12.4e} | {max_val:10.4f} | {overshoot:>14s}")
    print("-" * 75)

In [ ]:
# Gráficas de comparación para ambos regímenes
t_dense = np.linspace(0.0, T, 400)
x_dense = solucion_exacta(x0, C, t_dense)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Panel 1: dt = 0.1
ax = axes[0]
ax.plot(t_dense, x_dense, 'k-', label="Solución Exacta", zorder=1)
for name, color, style in [
    ("Euler Explícito", "#d95f02", "o--"),
    ("Euler Implícito", "#7570b3", "s--"),
    ("IMEX Linealizado", "#1b9e77", "^--")
]:
    t_num, x_num, _, _ = results[0.1][name]
    ax.plot(t_num, x_num, style, color=color, markersize=4, label=name, alpha=0.85, zorder=2)
ax.set_title("Régimen Estable: dt = 0.1 (dt * C = 0.2 < 2.0)")
ax.set_xlabel("Tiempo t")
ax.set_ylabel("Densidad poblacional x(t)")
ax.axhline(1.0, color="gray", linestyle=":", alpha=0.7)
ax.legend(frameon=True, loc="lower right")

# Panel 2: dt = 1.3
ax = axes[1]
ax.plot(t_dense, x_dense, 'k-', label="Solución Exacta", zorder=1)
for name, color, style in [
    ("Euler Explícito", "#d95f02", "o--"),
    ("Euler Implícito", "#7570b3", "s--"),
    ("IMEX Linealizado", "#1b9e77", "^--")
]:
    t_num, x_num, _, _ = results[1.3][name]
    ax.plot(t_num, x_num, style, color=color, markersize=6, label=name, alpha=0.9, zorder=2)
ax.set_title("Régimen Inestable: dt = 1.3 (dt * C = 2.6 > 2.0)")
ax.set_xlabel("Tiempo t")
ax.axhline(1.0, color="gray", linestyle=":", alpha=0.7)
ax.legend(frameon=True, loc="lower right")

plt.tight_layout()
plt.show()


## 7. Conclusiones y Síntesis Conceptual

1. **Sensibilidad del esquema explícito al paso temporal**:
   Cuando $\Delta t C = 0.2$, el método de Euler explícito reproduce la curva sigmoide con precisión. Sin embargo, al fijar $\Delta t = 1.3$ (donde $\Delta t C = 2.6 > 2.0$), el esquema explota localmente alrededor del punto de equilibrio $x^* = 1$, sobrepasando la cota física ($x \approx 1.22$) y exhibiendo oscilaciones no amortiguadas de período 2. Este fenómeno es puramente un artefacto de la discretización temporal y no representa la dinámica continua del modelo biológico.

2. **Robustez incondicional de los esquemas implícito e IMEX**:
   Tanto el método de Euler implícito como el esquema IMEX linealizado preservan la estabilidad cualitativa para $\Delta t = 1.3$. Ninguno de los dos presenta sobrepaso ($x \le 1.0$) ni oscilaciones espurias.

3. **Ventaja computacional del esquema IMEX**:
   El esquema IMEX logra la misma estabilidad asintótica que Euler implícito, pero sin requerir la resolución de ecuaciones cuadráticas en cada paso, ofreciendo una actualización analítica directa y positiva.
